# Assistente Médico Hospitalar — Pipeline RAG Completo
### Mistral-7B + QLoRA + LangChain RAG + FAISS + Prontuários

Este notebook executa o **pipeline completo** do assistente:

1. Verifica GPU disponível
2. Monta o Google Drive (logs de auditoria persistidos entre sessões)
3. Instala dependências
4. Clona o repositório do projeto (código-fonte + prontuários)
5. Carrega o modelo fine-tunado com quantização 4-bit
6. Constrói o índice FAISS com os prontuários fictícios
7. Processa consultas médicas com RAG, alertas clínicos e fontes estruturadas
8. Sobe interface Gradio com URL pública

> **Pré-requisito:** Execute em uma sessão do Google Colab com GPU (L4 ou T4).

## 1. Verificar GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU não detectada. Vá em Ambiente de execução → Alterar tipo de ambiente "
        "de execução → GPU (T4 ou L4)."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"✅ GPU: {gpu_name}")
print(f"   VRAM total: {vram_total:.1f} GB")

## 2. Montar Google Drive

O log de auditoria (`audit.log`) será salvo no Google Drive para **persistir entre sessões**.
Sem isso, o arquivo seria perdido ao encerrar o Colab.

O Colab solicitará permissão de acesso ao Drive na primeira execução.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Diretório de logs no Drive — será criado automaticamente se não existir
LOG_DIR = "/content/drive/MyDrive/assistente-medico/logs"
os.makedirs(LOG_DIR, exist_ok=True)

print(f"✅ Google Drive montado.")
print(f"   Logs de auditoria em: {LOG_DIR}")

## 3. Instalar Dependências

In [ ]:
%%time
# Instala dependências sem fixar versões para evitar conflitos com o ambiente do Colab.
# O Colab já fornece torch, numpy e transformers — apenas atualizamos se necessário.
!pip install -q -U \
    transformers \
    peft \
    accelerate \
    bitsandbytes \
    langchain \
    langchain-community \
    langchain-huggingface \
    langgraph \
    sentence-transformers \
    faiss-cpu \
    gradio

print("✅ Dependências instaladas.")

## ⚠️ Reiniciar a sessão antes de continuar

Após a instalação das dependências acima, é **obrigatório reiniciar o runtime** antes de prosseguir.
Isso garante que o numpy e outras bibliotecas atualizadas sejam carregadas corretamente na memória.

**Vá em: Ambiente de execução → Reiniciar sessão** (ou `Ctrl+M .`)

Após reiniciar, execute as células a partir da **seção 4** — a instalação não precisa ser repetida.

## 4. Clonar o Repositório

In [ ]:
REPO_URL = "https://github.com/rodrigoaraujorosa/fiap-ia-devs-8iadt-fase3-tech-challenge.git"  # ← altere aqui
REPO_DIR = "/content/assistente-medico"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repositório já clonado, atualizando...")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"✅ Diretório de trabalho: {os.getcwd()}")

## 5. Carregar Prontuários e Inicializar Logger

In [ ]:
%%time
import sys
sys.path.insert(0, REPO_DIR)

from src.database.prontuario_db import ProntuarioDB
from src.utils.logger import AuditLogger

# Carrega prontuários
db = ProntuarioDB("data/prontuarios.json")
db.load()
print(f"✅ Prontuários carregados: {db.total} registros")

# Logger aponta para o Google Drive — persiste entre sessões
logger = AuditLogger(log_dir=LOG_DIR)
print(f"✅ Logger de auditoria inicializado.")
print(f"   Arquivo: {LOG_DIR}/audit.log")

## 6. Carregar Modelo Fine-Tunado (QLoRA 4-bit)

In [ ]:
%%time
import warnings
import logging as _logging
warnings.filterwarnings("ignore")
_logging.getLogger("transformers").setLevel(_logging.ERROR)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig,
    pipeline as hf_pipeline,
)
from peft import PeftModel

BASE_MODEL_ID   = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_REPO_ID = "rodrigoaraujorosa/mistral-7b-assistente-hospitalar-v1"

# Quantização 4-bit NF4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Carregando modelo base...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Carregando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print(f"Aplicando adaptador LoRA: {ADAPTER_REPO_ID}")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO_ID)
model.eval()
model.generation_config = GenerationConfig(
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

# Pipeline HuggingFace para geração de texto
llm_pipeline = hf_pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.6,
    top_p=0.85,
    repetition_penalty=1.1,
)

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"\n✅ Modelo pronto! VRAM utilizada: {vram_used:.2f} GB")

## 7. Inicializar Pipeline RAG Completo

In [ ]:
%%time
from src.pipeline.retriever import ProntuarioRetriever
from src.pipeline.response_formatter import ResponseFormatter
from src.pipeline.prompt_templates import (
    MEDICAL_SYSTEM_PROMPT,
    MEDICAL_RAG_TEMPLATE,
    PRESCRIPTION_KEYWORDS,
    PRESCRIPTION_REFUSAL_MESSAGE,
    GENERIC_ERROR_MESSAGE,
)
from src.utils.models import MedicalResponse, ResponseSources
import unicodedata
from datetime import datetime, timezone

# Constrói índice FAISS com embeddings multilinguais
print("Construindo índice FAISS...")
retriever = ProntuarioRetriever(
    db=db,
    embedding_model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
)
retriever.build_index()
print("✅ Índice FAISS construído.")

formatter = ResponseFormatter()

def _normalize(text: str) -> str:
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in nfkd if not unicodedata.combining(c)).lower()

def consultar(consulta: str) -> MedicalResponse:
    """Processa uma consulta médica pelo pipeline RAG completo."""
    consulta = consulta.strip()
    if not consulta:
        raise ValueError("Consulta não pode ser vazia.")

    # 1. Detectar solicitação de prescrição
    consulta_norm = _normalize(consulta)
    if any(_normalize(kw) in consulta_norm for kw in PRESCRIPTION_KEYWORDS):
        sources = ResponseSources(dados_prontuario=[], conhecimento_modelo=False, paciente_encontrado=False)
        return MedicalResponse(
            resposta=PRESCRIPTION_REFUSAL_MESSAGE,
            fontes=sources,
            alertas=[],
            timestamp=datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            consulta_original=consulta,
        )

    # 2. Recuperar prontuários via RAG
    retrieved_docs = retriever.retrieve(consulta, k=3)
    paciente_encontrado = len(retrieved_docs) > 0

    # 3. Encontrar prontuário correspondente
    prontuario = None
    for doc in retrieved_docs:
        pid = doc.metadata.get("patient_id")
        if pid:
            prontuario = db._index_by_id.get(pid)
            if prontuario:
                break

    # 4. Montar prompt e invocar LLM
    if retrieved_docs and paciente_encontrado:
        context = "\n\n---\n\n".join(doc.page_content for doc in retrieved_docs)
    else:
        context = "Nenhum prontuário encontrado para o paciente mencionado."

    prompt = (
        f"{MEDICAL_SYSTEM_PROMPT}\n\n"
        + MEDICAL_RAG_TEMPLATE.format(context=context, question=consulta)
    )

    result = llm_pipeline(prompt)
    generated = result[0].get("generated_text", "")
    if generated.startswith(prompt):
        generated = generated[len(prompt):].strip()
    raw_response = generated or "Não foi possível gerar uma resposta."

    # 5. Formatar resposta com fontes e alertas
    response = formatter.format(
        raw_response=raw_response,
        retrieved_docs=retrieved_docs,
        consulta=consulta,
        prontuario=prontuario,
        paciente_encontrado=paciente_encontrado,
    )

    # 6. Registrar no log de auditoria (Google Drive)
    logger.log_interaction(consulta=consulta, resposta=response, sources=response.fontes)

    return response

print("✅ Pipeline RAG pronto para consultas.")

## 8. Testar o Pipeline (Consultas de Exemplo)

In [ ]:
# Consulta 1: paciente com exames pendentes
resp = consultar("Quais exames estão pendentes para Ana Clara Ferreira?")

print("=" * 70)
print("CONSULTA:", resp.consulta_original)
print("=" * 70)
print(resp.resposta)

if resp.alertas:
    print("\n🚨 ALERTAS CLÍNICOS:")
    for alerta in resp.alertas:
        print(f"  [{alerta.prioridade.value}] {alerta.tipo.value}: {alerta.descricao}")
        print(f"  Exames: {', '.join(alerta.exames_afetados)}")

In [ ]:
# Consulta 2: consulta genérica (fallback para conhecimento do modelo)
resp2 = consultar("Qual o tratamento para hipertensão arterial sistêmica?")

print("=" * 70)
print("CONSULTA:", resp2.consulta_original)
print("=" * 70)
print(resp2.resposta)
print("\nPaciente encontrado:", resp2.fontes.paciente_encontrado)

In [ ]:
# Consulta 3: solicitação de prescrição (deve ser recusada)
resp3 = consultar("Gere uma prescrição médica para o paciente João Silva.")

print("=" * 70)
print("CONSULTA:", resp3.consulta_original)
print("=" * 70)
print(resp3.resposta)

In [ ]:
# Verificar log salvo no Drive
log_path = f"{LOG_DIR}/audit.log"
if os.path.exists(log_path):
    with open(log_path, encoding="utf-8") as f:
        linhas = f.readlines()
    print(f"📋 {len(linhas)} entrada(s) no log de auditoria ({log_path}):")
    for linha in linhas[-3:]:  # exibe as 3 últimas entradas
        import json
        entry = json.loads(linha)
        print(f"  [{entry['timestamp']}] {entry['event_type']} — {entry.get('consulta', '')[:60]}")
else:
    print("Log ainda não criado.")

## 9. Interface Gradio com URL Pública

A célula abaixo sobe uma interface interativa com **URL pública válida por 72 horas**.

In [ ]:
import gradio as gr

def gradio_consultar(consulta: str) -> tuple[str, str, str]:
    """Wrapper para a interface Gradio. Retorna (resposta, alertas, fontes)."""
    if not consulta.strip():
        return "Por favor, digite uma consulta.", "", ""
    try:
        resp = consultar(consulta)

        # Formata alertas
        if resp.alertas:
            alertas_txt = "\n".join(
                f"🚨 [{a.prioridade.value}] {a.tipo.value}\n"
                f"   {a.descricao}\n"
                f"   Exames: {', '.join(a.exames_afetados)}"
                for a in resp.alertas
            )
        else:
            alertas_txt = "Nenhum alerta clínico identificado."

        # Formata fontes
        fontes = resp.fontes
        fontes_txt = (
            f"Paciente encontrado: {'Sim' if fontes.paciente_encontrado else 'Não'}\n"
            f"Dados do prontuário (RAG): {', '.join(fontes.dados_prontuario) or 'nenhum'}\n"
            f"Conhecimento do modelo: {'Sim' if fontes.conhecimento_modelo else 'Não'}\n"
            f"Timestamp: {resp.timestamp}"
        )

        return resp.resposta, alertas_txt, fontes_txt

    except Exception as exc:
        logger.log_error(exc, context={"consulta": consulta})
        return GENERIC_ERROR_MESSAGE, "", ""


exemplos = [
    ["Quais exames estão pendentes para Ana Clara Ferreira?"],
    ["Qual o estado atual dos exames do paciente Carlos Eduardo Mendes?"],
    ["Quais pacientes têm exames com resultado alterado?"],
    ["Qual o tratamento recomendado para anemia ferropriva?"],
    ["Gere uma prescrição médica para o paciente João Silva."],
]

with gr.Blocks(theme=gr.themes.Soft(), title="Assistente Médico Hospitalar — RAG") as demo:

    gr.Markdown(
        "# 🏥 Assistente Médico Hospitalar\n"
        "**Mistral-7B-Instruct-v0.2 + QLoRA + LangChain RAG + FAISS**\n\n"
        "> ⚠️ Este sistema é destinado a fins educacionais e de pesquisa. "
        "Não deve ser utilizado para diagnóstico ou decisão clínica real."
    )

    with gr.Row():
        with gr.Column(scale=2):
            consulta_input = gr.Textbox(
                label="Consulta Médica",
                placeholder="Ex: Quais exames estão pendentes para Ana Clara Ferreira?",
                lines=3,
            )
            with gr.Row():
                limpar_btn    = gr.Button("Limpar", variant="secondary")
                consultar_btn = gr.Button("Consultar", variant="primary")

            resposta_output = gr.Textbox(
                label="Resposta do Assistente",
                lines=12,
                interactive=False,
            )

        with gr.Column(scale=1):
            alertas_output = gr.Textbox(
                label="🚨 Alertas Clínicos",
                lines=6,
                interactive=False,
            )
            fontes_output = gr.Textbox(
                label="📋 Fontes da Resposta",
                lines=6,
                interactive=False,
            )

    gr.Markdown("### Exemplos de Consultas")
    gr.Examples(
        examples=exemplos,
        inputs=consulta_input,
        label="Clique para carregar um exemplo",
    )

    gr.Markdown(
        "---\n"
        "Modelo: `rodrigoaraujorosa/mistral-7b-assistente-hospitalar-v1` | "
        "Tech Challenge — Fase 3 — 8IADT"
    )

    consultar_btn.click(
        fn=gradio_consultar,
        inputs=consulta_input,
        outputs=[resposta_output, alertas_output, fontes_output],
    )
    limpar_btn.click(
        fn=lambda: ("", "", "", ""),
        outputs=[consulta_input, resposta_output, alertas_output, fontes_output],
    )
    consulta_input.submit(
        fn=gradio_consultar,
        inputs=consulta_input,
        outputs=[resposta_output, alertas_output, fontes_output],
    )

demo.launch(share=True, show_error=True)